In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Tuple, Dict

# 固定亂數種子，讓每次結果更容易重現
np.random.seed(7)

In [ ]:
def random_orthogonal(n: int) -> np.ndarray:
    """產生一個 n x n 的隨機正交矩陣。"""
    A = np.random.randn(n, n)
    Q, R = np.linalg.qr(A)
    signs = np.sign(np.diag(R))
    signs[signs == 0] = 1.0
    return Q @ np.diag(signs)


def make_psd_ground_truth(n: int, eigenvalues: List[float]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """建立對稱 PSD ground truth：W_hat = U diag(lambdas) U^T。"""
    if len(eigenvalues) > n:
        raise ValueError("eigenvalues 的長度不能超過 n")

    lambdas_full = np.zeros(n)
    lambdas_full[:len(eigenvalues)] = np.array(eigenvalues, dtype=float)

    U = random_orthogonal(n)
    W_hat = U @ np.diag(lambdas_full) @ U.T
    W_hat = 0.5 * (W_hat + W_hat.T) #保障一定是對稱
    return W_hat, U, lambdas_full

def spectral_norm(W: np.ndarray) -> float:
    """回傳矩陣的 spectral norm，也就是最大奇異值。"""
    return float(np.linalg.eigvalsh(W)[-1])


def nuclear_norm(W: np.ndarray) -> float:
    """回傳矩陣的 nuclear norm，也就是所有奇異值總和。"""
    return float(np.sum(np.linalg.eigvalsh(W)))


def effective_rank(W: np.ndarray, eps: float = 1e-12) -> float:
    """計算 effective rank：r(W) = ||W||_* / ||W||。適用於對稱 PSD 矩陣。"""
    evals = np.linalg.eigvalsh(W)
    top = float(evals[-1])

    if top < eps:
        return 0.0

    return float(np.sum(evals) / top)

In [ ]:
@dataclass
class GDResult:
    errors: np.ndarray
    effective_ranks: np.ndarray
    Ws: List[np.ndarray]


def run_gd_matrix_factorization(W_hat: np.ndarray, alpha: float, eta: float, steps: int) -> GDResult:
    """跑 N=2 的 gradient descent。"""
    n = W_hat.shape[0]
    W1 = alpha * np.eye(n)
    W2 = alpha * np.eye(n)

    errors = []
    eff_ranks = []
    Ws = []

    for _ in range(steps + 1):
        W = W2 @ W1
        E = W - W_hat

        errors.append(np.linalg.norm(E, ord="fro"))
        eff_ranks.append(effective_rank(W))
        Ws.append(W.copy())

        grad_W1 = W2.T @ E
        grad_W2 = E @ W1.T

        W1 = W1 - eta * grad_W1
        W2 = W2 - eta * grad_W2

    return GDResult(errors=np.array(errors), effective_ranks=np.array(eff_ranks), Ws=Ws)

In [ ]:
def g_lambda_alpha(t: np.ndarray, lam: float, alpha: float) -> np.ndarray:
    """N=2 時的 g_{lambda,alpha}(t)。"""
    return 1.0 + (lam / (alpha ** 2) - 1.0) * np.exp(-2.0 * lam * t)


def effective_rank_of_truncation(lambdas: np.ndarray, L: int) -> float:
    """計算 r(W_hat_L)。"""
    if L < 1:
        raise ValueError("L 必須至少為 1")
    if lambdas[0] <= 0:
        return 0.0
    return float(np.sum(lambdas[:L]) / lambdas[0])


def theorem31_windows(lambdas: np.ndarray, alpha: float, eta: float, steps: int, eps: float, C: float, L_values: List[int]) -> Dict[int, List[Tuple[int, int]]]:
    """對每個 L，找出離散時間 k 上滿足 I1 ∩ I2 ∩ I3 的區間。"""
    t_grid = eta * np.arange(steps + 1)
    lambda1 = lambdas[0]
    L_prime = int(np.sum(lambdas > alpha ** 2))
    windows = {}

    for L in L_values:
        if L > L_prime or L < 1:
            windows[L] = []
            continue

        r_hat_L = effective_rank_of_truncation(lambdas, L)
        g1 = g_lambda_alpha(t_grid, lambda1, alpha)

        mask_I1 = np.ones_like(t_grid, dtype=bool)
        if L > 1:
            ratios = []
            for ell in range(1, L):
                gell = g_lambda_alpha(t_grid, lambdas[ell], alpha)
                ratios.append(np.abs(g1 / gell - 1.0))
            max_ratio = np.max(np.vstack(ratios), axis=0)
            mask_I1 = max_ratio < eps

        if L < L_prime:
            gLp1 = g_lambda_alpha(t_grid, lambdas[L], alpha)
            lhs_I2 = np.abs((g1 / gLp1) * (lambdas[L] / lambda1))
            rhs_I2 = (r_hat_L / max(L_prime - L, 1)) * eps
            mask_I2 = lhs_I2 < rhs_I2
        else:
            mask_I2 = np.ones_like(t_grid, dtype=bool)

        mask_I3 = g1 < C
        mask = mask_I1 & mask_I2 & mask_I3

        intervals = []
        start = None
        for k, val in enumerate(mask):
            if val and start is None:
                start = k
            elif (not val) and start is not None:
                intervals.append((start, k - 1))
                start = None
        if start is not None:
            intervals.append((start, len(mask) - 1))

        windows[L] = intervals

    #Output:{1: [(20, 90)],2: [(100, 180)],3: [(220, 350)]}
    #L=1 的 plateau 預測區間是 k=20 到 90....
    return windows


In [ ]:
def best_rank_L_effective_ranks(lambdas: np.ndarray, max_L: int) -> Dict[int, float]:
    """回傳 L=1,...,max_L 對應的 r(W_hat_L)。"""
    return {L: effective_rank_of_truncation(lambdas, L) for L in range(1, max_L + 1)}


def add_windows(ax, windows: Dict[int, List[Tuple[int, int]]], colors: Dict[int, str], alpha_fill: float = 0.28): #alpha_fill 透明度
    """在圖上加上 theorem window 的藍色陰影。"""
    for L, intervals in windows.items():
        for i, (s, e) in enumerate(intervals):
            label = rf"$\bigcap_j \quad I_j$ for $L={L}$" if i == 0 else None
            ax.axvspan(s, e, color=colors[L], alpha=alpha_fill, label=label)


def plot_case_2x2(clean_res: GDResult, noisy_res: GDResult, lambdas_clean: np.ndarray, windows_clean: Dict[int, List[Tuple[int, int]]], title_prefix: str = "", max_L_shown: int = 3):
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    colors = {1: '#2F3EEA', 2: '#5160F0', 3: '#7B88F5', 4: '#9CA5FA', 5: '#B2B8FB', 6: '#C7CCFC'}
    ks = np.arange(len(clean_res.errors))

    ax = axes[0, 0]
    ax.plot(ks, clean_res.errors, color='black', linewidth=2.2)
    ax.set_yscale('log')
    add_windows(ax, {L: windows_clean.get(L, []) for L in range(1, max_L_shown + 1)}, colors)
    ax.set_title('Approximation Error')
    ax.set_xlabel('Iteration (k)')
    ax.set_ylabel(r'$\|W(k)-\widehat W\|_F$')
    ax.legend(loc='upper right', fontsize=10)
    ax.text(0.5, -0.23, '(a) Approximation error without noise.', transform=ax.transAxes, ha='center', fontsize=13)

    ax = axes[0, 1]
    ax.plot(ks, clean_res.effective_ranks, color='black', linewidth=2.2)
    add_windows(ax, {L: windows_clean.get(L, []) for L in range(1, max_L_shown + 1)}, colors)
    ax.set_title('Effective Rank')
    ax.set_xlabel('Iteration (k)')
    ax.set_ylabel(r'$r(W(k))$')
    ax.set_ylim(0, 3)
    trunc_ranks = best_rank_L_effective_ranks(lambdas_clean, max_L_shown)
    for L, rval in trunc_ranks.items():
        ax.axhline(rval, color='gray', linestyle='--', linewidth=0.9)
        ax.text(0.72 * ks[-1], rval + 0.02, rf'$r(\widehat W_{{{L}}})$', fontsize=11)
    ax.legend(loc='upper right', fontsize=10)
    ax.text(0.5, -0.23, '(b) Effective rank without noise.', transform=ax.transAxes, ha='center', fontsize=13)

    ax = axes[1, 0]
    ax.plot(ks, noisy_res.errors, color='black', linewidth=2.2)
    ax.set_yscale('log')
    add_windows(ax, {L: windows_clean.get(L, []) for L in range(1, max_L_shown + 1)}, colors)
    ax.set_title('Approximation Error')
    ax.set_xlabel('Iteration (k)')
    ax.set_ylabel(r'$\|W(k)-\widehat W\|_F$')
    ax.legend(loc='upper right', fontsize=10)
    ax.text(0.5, -0.23, '(c) Approximation error with noise.', transform=ax.transAxes, ha='center', fontsize=13)

    ax = axes[1, 1]
    ax.plot(ks, noisy_res.effective_ranks, color='black', linewidth=2.2)
    add_windows(ax, {L: windows_clean.get(L, []) for L in range(1, max_L_shown + 1)}, colors)
    ax.set_title('Effective Rank')
    ax.set_xlabel('Iteration (k)')
    ax.set_ylabel(r'$r(W(k))$')
    ax.set_ylim(0, 3)
    for L, rval in trunc_ranks.items():
        ax.axhline(rval, color='gray', linestyle='--', linewidth=0.9)
        ax.text(0.72 * ks[-1], rval + 0.02, rf'$r(\widehat W_{{{L}}})$', fontsize=11)
    ax.legend(loc='upper right', fontsize=10)
    ax.text(0.5, -0.23, '(d) Effective rank with noise.', transform=ax.transAxes, ha='center', fontsize=13)

    if title_prefix:
        fig.suptitle(title_prefix, fontsize=16)

    plt.tight_layout()
    return fig

In [ ]:
@dataclass
class ExperimentConfig:
    n: int
    eigenvalues: List[float]
    alpha: float
    eta: float
    steps: int
    noise_ratio: float
    eps_theorem: float
    C_theorem: float
    title: str


def add_symmetric_noise(W: np.ndarray, noise_ratio: float) -> np.ndarray:
    """加入對稱 Gaussian noise，並把 operator norm 調整成 noise_ratio * ||W||。"""
    n = W.shape[0]
    Xi = np.random.randn(n, n)
    Xi = 0.5 * (Xi + Xi.T)
    Xi_norm = spectral_norm(Xi)
    target_norm = noise_ratio * spectral_norm(W)
    if Xi_norm > 0:
        Xi = Xi * (target_norm / Xi_norm)
    return Xi


def run_full_experiment(cfg: ExperimentConfig):
    W_clean, U, lambdas = make_psd_ground_truth(cfg.n, cfg.eigenvalues)
    Xi = add_symmetric_noise(W_clean, cfg.noise_ratio)
    W_noisy = 0.5 * ((W_clean + Xi) + (W_clean + Xi).T)

    res_clean = run_gd_matrix_factorization(W_clean, cfg.alpha, cfg.eta, cfg.steps)
    res_noisy = run_gd_matrix_factorization(W_noisy, cfg.alpha, cfg.eta, cfg.steps)

    L_values = list(range(1, min(len(cfg.eigenvalues), 6) + 1))
    windows = theorem31_windows(
        lambdas=lambdas,
        alpha=cfg.alpha,
        eta=cfg.eta,
        steps=cfg.steps,
        eps=cfg.eps_theorem,
        C=cfg.C_theorem,
        L_values=L_values,
    )

    return {
        'W_clean': W_clean,
        'W_noisy': W_noisy,
        'Xi': Xi,
        'lambdas': lambdas,
        'res_clean': res_clean,
        'res_noisy': res_noisy,
        'windows': windows,
    }

In [ ]:
base_cfg = ExperimentConfig(
    n=200,
    eigenvalues=[10.0, 5.0, 1.0],
    alpha=1e-2,
    eta=1e-2,
    steps=1000,
    noise_ratio=0.05,
    eps_theorem=2.2e-2,
    C_theorem=17.0,
    title='Case A: base setting (n=200, rank=3)',
)

base_out = run_full_experiment(base_cfg)

fig = plot_case_2x2(
    base_out['res_clean'],
    base_out['res_noisy'],
    base_out['lambdas'],
    base_out['windows'],
    title_prefix=base_cfg.title,
    max_L_shown=3,
)
plt.show()